# Cell-table information map and virtual tumour state (Danenberg + Jackson IMC)

Plan: `future_plan.md` §14 (Steps 0–1) and §15-lite. CPU only; no images and no training of image models.

**Step 0: immune–tumour contact check.**
- Per image, count contacts between epithelial (tumour) and immune cells in the published contact graph.
- Compare with 200 permutations of the broad-class labels within the image (positions and graph fixed).
- Report the share of images enriched, depleted or neither (permutation p < 0.05), and whether the patient-level ratio follows subtype, ER or grade.

**Step 1: per-marker information map, full panel of each cohort** (36 Danenberg, 32 Jackson, plus H3K27me3/H3).
- For each marker M, predict each cell's M (arcsinh) with gradient boosting from:
  - **own:** the cell's other channels (including H3, DNA1, DNA2) and log area;
  - **own+ring:** plus the mean of non-touching cells within 30 µm (centroid distance), for every channel except M, and their count;
  - **own+ring+contact:** plus the mean of touching cells, for every channel except M, and their count.
- M is never used from any cell. An image model that imputes M doesn't see it anywhere either.
- CV: 5-fold, grouped by patient. Training uses 150k random cells per fold; every held-out cell is predicted.
- Readouts:
  - cell-level R² (redundancy);
  - ΔR² (context);
  - within-image r;
  - 1000× patient bootstrap CIs.
  - Signal quality: the share of cells with robust z > 3, and p99 − p50.

**Gate 1 and §15-lite: virtual tumour state.**
- In Jackson, predict every Jackson-only marker from only the channels Danenberg also has: the 11 shared markers + H3, DNA1, DNA2.
- Inputs are standardised per cohort (median, p5–p95). This uses Danenberg input statistics only, never target statistics; it is level B, labelled.
- **Gate 1 (fixed before the run):** a marker passes if its cell-level R² (own) has a patient-bootstrap CI lower bound > 0.
- Markers that pass are imputed into every Danenberg cell with a model trained on Jackson.
- **Patient feature:** the mean virtual value over tumour (epithelial) cells, for patients with ≥ 50 tumour cells.
- **Patient-level validation against Danenberg labels.** The gain over a logistic model on the patient means of the 14 input channels is what counts.
  - PR status (all patients, and ER+ only) from virtual PR;
  - PAM50 basal-like from virtual CK14 and EGFR;
  - LumA vs LumB from virtual GATA3 and PR.
- **Survival** (Danenberg DSS; covariates age, grade, log size, log positive nodes, ER, HER2):
  - **T1:** the adjusted HR per SD of the virtual marker. Compared with the HR of the measured marker in Jackson Basel (covariates age, grade, log size, N+) and with literature priors.
  - **T2:** ΔC-index of clinical + X over clinical.
  - **T3:** ΔC-index of clinical + inputs + X over clinical + inputs.
  - Out-of-fold C-index from 5 × 5-fold CV, with a paired patient-bootstrap CI. BH correction across markers.

**Calibration: cross-cohort cost.**
- Each of the 11 shared markers is treated as a pseudo-target: it is predicted from the other 10 + nuclear channels.
- The model is trained in Jackson and scored in Danenberg against the measured value, next to its within-Jackson CV score.

**Context gain (GK input, fixed before the run):**
- ΔR² (own+ring+contact − own) with a CI lower bound > 0.
- It is flagged "contact-only" when ΔR² (own+ring − own) does not also exclude 0. Contact-only gains can come from spillover between touching cells.

In [ ]:
import os, glob, json, time, sys, subprocess, warnings
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
from scipy.spatial import cKDTree
from scipy.stats import spearmanr, kruskal, mannwhitneyu
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
try:
    import lifelines
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lifelines"], check=True)
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1"          # tiny settings for a local test on synthetic data
SEED = 20261012
COFACTOR = 1.0
MIN_CELLS = 200             # per image, as in earlier runs
MIN_TUM = 50                # tumour cells per patient for patient features
RING_R = 30.0               # µm (= px)
N_FOLD = 5
N_TRAIN = 20_000 if SMOKE else 150_000
N_TRAIN_FINAL = 30_000 if SMOKE else 300_000
N_BOOT = 100 if SMOKE else 1000
N_PERM = 50 if SMOKE else 200
N_REP = 2 if SMOKE else 5
N_JOBS = os.cpu_count()
LGB = dict(n_estimators=30 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=N_JOBS, verbose=-1)
NUC = ["histone_h3", "dna1", "dna2"]
SHARED = {   # Jackson name -> Danenberg name (the 11 shared markers; HER2 is clone 3B5 in both)
    "estrogen_receptor_alpha": "estrogen_receptor_alpha", "c_erb_b_2_her2": "c_erb_b_2_her2_3b5",
    "ki_67": "ki_67", "cd3": "cd3", "cd20": "cd20", "cd45": "cd45", "cd68": "cd68",
    "cytokeratin_5": "cytokeratin_5", "cytokeratin_8_18": "cytokeratin_8_18",
    "pan_cytokeratin": "pan_cytokeratin", "sma": "sma"}
K27, K27R = "histone_h3_trimethylate", "h3k27me3_per_h3"
GROUPS = {
    "danenberg": {
        "clinical": ["estrogen_receptor_alpha", "c_erb_b_2_her2_3b5", "c_erb_b_2_her2_d8f12"],
        "proliferation": ["ki_67"],
        "cell death": ["cleaved_caspase3"],
        "antigen presentation": ["hla_abc", "beta_2_microglobulin", "hla_dr"],
        "immune state": ["cd279_pd_1", "cd278_icos", "cd134", "gitr_tnfrsf18", "cd38", "cd57", "cd45ra"],
        "immune lineage": ["cd45", "cd3", "cd4", "cd8a", "cd20", "cd68", "cd163", "cd11c", "cd15", "cd16", "foxp3"],
        "epithelial lineage": ["pan_cytokeratin", "cytokeratin_8_18", "cytokeratin_5"],
        "stroma and vessels": ["sma", "fsp1", "cd140b_pdgf_receptor_beta", "podoplanin", "caveolin_1", "cd31_v_wf",
                               "cxcl12_sdf_1"]},
    "jackson": {
        "clinical": ["estrogen_receptor_alpha", "progesterone_receptor_a_b", "c_erb_b_2_her2"],
        "proliferation": ["ki_67", "histone_h3_phospho"],
        "cell death": ["cleaved_parp"],
        "epigenetic": [K27, K27R],
        "tumour state": ["s6", "m_tor", "c_myc", "egfr", "p53", "twist", "slug", "cd44", "carbonic_anhydrase_ix"],
        "immune lineage": ["cd45", "cd3", "cd20", "cd68"],
        "epithelial lineage": ["pan_cytokeratin", "cytokeratin_8_18", "cytokeratin_5", "cytokeratin_7",
                               "keratin_14_krt14", "cytokeratin_19", "e_cadherin_p_cadherin", "gata3"],
        "stroma and vessels": ["sma", "vimentin", "fibronectin", "v_wf"]}}
GROUP_OF = {k: {m: g for g, ms in v.items() for m in ms} for k, v in GROUPS.items()}
LITE_TARGETS = ["carbonic_anhydrase_ix", "p53", "c_myc", "s6", "m_tor", "egfr", "twist", "slug", "cd44", K27, K27R,
                "progesterone_receptor_a_b", "gata3", "keratin_14_krt14", "cytokeratin_7", "cytokeratin_19"]
# Expected sign of log HR per SD for DSS, from the literature (verify each before citing). None = no prior.
PRIOR_SIGN = {"carbonic_anhydrase_ix": 1, "p53": 1, "gata3": -1, "progesterone_receptor_a_b": -1, K27: -1, K27R: -1}
STROMAL_KEYS = ("fibro", "endothel", "stroma", "pdpn", "myofib", "vessel", "vwf")
T0 = time.time()

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def num(s):
    return pd.to_numeric(pd.Series(s).replace({"nan": np.nan, "[]": np.nan}), errors="coerce")

def near(name, anchor):
    hits = glob.glob(f"{IN}/**/{name}", recursive=True)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

def locate():
    out = {}
    for p in sorted(glob.glob(f"{IN}/**/cells.parquet", recursive=True)):
        key = "danenberg" if "metabric_id" in pq.read_schema(p).names else "jackson"
        assert key not in out, (key, p)
        out[key] = dict(cells=p, report=near("prep_report.json", p), contacts=near("cell_neighbours.parquet", p),
                        clinical=near("clinical.parquet", p))
        if key == "danenberg":
            out[key]["split"] = near("danenberg_patient_split.csv", p)
    print(json.dumps(out, indent=1))
    assert set(out) == {"danenberg", "jackson"}
    return out

## Load both cohorts (cells, contact graph, clinical)

In [ ]:
def load(key, P):
    rep = json.load(open(P["report"]))
    chan = list(rep["channel_map"].values()) if key == "danenberg" else list(rep["channel_map"])
    cells = pd.read_parquet(P["cells"])
    print(key, cells.shape, "columns:", list(cells.columns))
    if key == "danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid = "metabric_id"
    else:
        cells = cells[cells.is_tumour == 1]
        pid = "patient_id"
    n_per = cells.groupby("image_name").size()
    cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    if key == "danenberg":
        epi_frac = cells.groupby("cell_phenotype").is_epithelial.mean()
        bmap = {p: "epithelial" if epi_frac[p] > 0.5 else
                "stromal" if any(s in p.lower() for s in STROMAL_KEYS) else "immune" for p in epi_frac.index}
        broad = cells.cell_phenotype.map(bmap).astype(str).values
        epi = cells.is_epithelial.values == 1
    else:
        broad = cells.broad_class.astype(str).values
        epi = broad == "epithelial"
    img_code, img_names = pd.factorize(cells.image_name, sort=True)
    starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
    A = np.arcsinh(cells[chan].values.astype(np.float64) / COFACTOR).astype(np.float32)
    loc = pd.Series(np.arange(len(cells)), index=pd.MultiIndex.from_arrays(
        [cells.image_number.values, cells.object_number.values]))
    con = pd.read_parquet(P["contacts"])
    ca = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["from"].values])).values
    cb = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["to"].values])).values
    ok = ~(np.isnan(ca) | np.isnan(cb))
    ca, cb = ca[ok].astype(np.int64), cb[ok].astype(np.int64)
    ca, cb = ca[ca != cb], cb[ca != cb]
    assert (img_code[ca] == img_code[cb]).all()
    D = dict(key=key, chan=chan, protein=[c for c in chan if c not in NUC], A=A, img=img_code,
             img_names=np.asarray(img_names), starts=starts, patient=cells[pid].astype(str).values,
             broad=broad, epi=epi, xy=cells[["x", "y"]].values.astype(np.float64), ca=ca, cb=cb,
             log_area=np.log(np.maximum(cells["area"].values, 1)).astype(np.float32) if "area" in cells else None,
             cohort=cells["cohort"].astype(str).values if "cohort" in cells else None)
    missing = [m for m in D["protein"] if m not in GROUP_OF[key]]
    assert not missing, f"{key}: markers without a group {missing}"
    print(f"{key}: {len(cells):,} cells, {len(img_names)} images, {len(np.unique(D['patient']))} patients, "
          f"{epi.mean():.2f} epithelial, broad {pd.Series(broad).value_counts().to_dict()}, "
          f"{len(ca):,} contact rows mapped, area {'yes' if D['log_area'] is not None else 'no'}")
    return D

def clinical_danenberg(P):
    c = pd.read_parquet(P["clinical"]).replace({"nan": np.nan})
    c["pid"] = c.metabric_id.astype(str)
    c = c.set_index("pid")
    split = pd.read_csv(P["split"]).set_index("metabric_id")
    t = pd.DataFrame(index=c.index)
    t["pam50"] = c.CLAUDIN_SUBTYPE.fillna(pd.Series(c.index, index=c.index).map(split.pam50))
    t["er"], t["pr"], t["her2"] = c.ER_STATUS, c.PR_STATUS, c.HER2_STATUS
    g = num(c.GRADE).values
    ln = num(c.LYMPH_NODES_EXAMINED_POSITIVE).values
    vs = c.VITAL_STATUS
    t["dss_time"] = num(c.OS_MONTHS).values
    t["dss_event"] = np.where(vs.isna(), np.nan, (vs == "Died of Disease").astype(float))
    t["grade"] = g
    cov = pd.DataFrame(index=c.index)
    cov["age"] = num(c.AGE_AT_DIAGNOSIS).values
    cov["grade"] = g
    cov["log_size"] = np.log1p(num(c.TUMOR_SIZE).values)
    cov["log_nodes"] = np.log1p(ln)
    cov["er_pos"] = np.where(c.ER_STATUS.isna(), np.nan, (c.ER_STATUS == "Positive").astype(float))
    cov["her2_pos"] = np.where(c.HER2_STATUS.isna(), np.nan, (c.HER2_STATUS == "Positive").astype(float))
    return t, cov

def clinical_jackson(P):
    c = pd.read_parquet(P["clinical"]).replace({"nan": np.nan, "[]": np.nan})
    c = c.groupby("patient_id").first()
    c.index = c.index.astype(str)
    t = pd.DataFrame(index=c.index)
    t["clinical_type"] = c.clinical_type
    g = num(c.grade).values                                # METASTASIS -> NaN
    t["grade"] = g
    t["er"], t["pr"], t["her2"] = c.ERStatus, c.PRStatus, c.HER2Status
    basel = (c.cohort == "basel").values
    st = c.Patientstatus
    ok = basel & st.notna().values
    t["dss_time"] = np.where(basel, num(c.OSmonth).values, np.nan)
    t["dss_event"] = np.where(ok, (st == "death by primary disease").astype(float), np.nan)
    cov = pd.DataFrame(index=c.index)
    cov["age"] = num(c.age).values
    cov["grade"] = g
    cov["log_size"] = np.log1p(num(c.tumor_size).values)
    n = c.PTNM_N.astype(str).str.strip().str.lower().str.lstrip("n")      # Basel "0", "1a"; Zurich "n0", "n1"
    cov["n_pos"] = np.where(n.str.startswith("0"), 0.0, np.where(n.str.match(r"^[123]"), 1.0, np.nan))
    return t, cov

PATHS = locate()
DATA = {k: load(k, PATHS[k]) for k in ("danenberg", "jackson")}
CLIN = {"danenberg": clinical_danenberg(PATHS["danenberg"]), "jackson": clinical_jackson(PATHS["jackson"])}
for k, (t, cov) in CLIN.items():
    pats = np.unique(DATA[k]["patient"])
    print(k, "patients with clinical rows:", t.index.isin(pats).sum(), "of", len(pats))
print("Danenberg PAM50:", CLIN["danenberg"][0].pam50.value_counts(dropna=False).to_dict())
tick("loaded")

## Step 0: immune–tumour contact check

In [ ]:
def contact_check(D, rng):
    lab = pd.Series(D["broad"]).map({"epithelial": 0, "immune": 1, "stromal": 2}).values
    n = len(lab)
    key = np.unique(np.minimum(D["ca"], D["cb"]) * n + np.maximum(D["ca"], D["cb"]))
    pa, pb = key // n, key % n
    ip = D["img"][pa]
    o = np.argsort(ip, kind="stable")
    pa, pb, ip = pa[o], pb[o], ip[o]
    bnd = np.searchsorted(ip, np.arange(len(D["img_names"]) + 1))
    rows = []
    for i in range(len(D["img_names"])):
        s, e = D["starts"][i], D["starts"][i + 1]
        la, lb = pa[bnd[i]:bnd[i + 1]] - s, pb[bnd[i]:bnd[i + 1]] - s
        li = lab[s:e]
        m, n_e, n_i = len(li), int((li == 0).sum()), int((li == 1).sum())
        row = dict(cohort=D["key"], image=D["img_names"][i], patient=D["patient"][s], n_cells=m,
                   frac_epithelial=n_e / m, frac_immune=n_i / m, n_pairs=len(la))
        if len(la) and n_e and n_i:
            obs = int((((li[la] == 0) & (li[lb] == 1)) | ((li[la] == 1) & (li[lb] == 0))).sum())
            P = rng.permuted(np.tile(li, (N_PERM, 1)), axis=1)
            perm = (((P[:, la] == 0) & (P[:, lb] == 1)) | ((P[:, la] == 1) & (P[:, lb] == 0))).sum(1)
            exp = len(la) * 2 * n_e * n_i / (m * (m - 1))
            p = (1 + np.sum(np.abs(perm - exp) >= abs(obs - exp))) / (N_PERM + 1)
            row.update(obs=obs, exp=exp, log2_ratio=np.log2((obs + 0.5) / (exp + 0.5)),
                       z=(obs - perm.mean()) / (perm.std() + 1e-9), p=p,
                       call="enriched" if (p < 0.05 and obs > exp) else "depleted" if (p < 0.05 and obs < exp) else "neither")
        rows.append(row)
    return pd.DataFrame(rows)

rng = np.random.default_rng(SEED)
step0 = pd.concat([contact_check(D, rng) for D in DATA.values()], ignore_index=True)
step0.to_csv(f"{OUT}/step0_contact_per_image.csv", index=False)
S0 = {}
for key, g in step0.dropna(subset=["log2_ratio"]).groupby("cohort"):
    t = CLIN[key][0]
    pp = g.groupby("patient").log2_ratio.mean()
    tt = t.reindex(pp.index)
    sub = "pam50" if key == "danenberg" else "clinical_type"
    groups = [pp[tt[sub] == v].values for v in tt[sub].dropna().unique() if (tt[sub] == v).sum() >= 10]
    er_pos, er_neg = pp[tt.er.astype(str).str.lower().str.startswith("pos")], pp[tt.er.astype(str).str.lower().str.startswith("neg")]
    gm = tt.grade.notna()
    S0[key] = dict(
        n_images=len(g), share_enriched=float((g.call == "enriched").mean()),
        share_depleted=float((g.call == "depleted").mean()), share_neither=float((g.call == "neither").mean()),
        median_log2_ratio_image=float(g.log2_ratio.median()), n_patients=len(pp),
        median_log2_ratio_patient=float(pp.median()),
        subtype_kruskal_p=float(kruskal(*groups).pvalue) if len(groups) > 1 else None,
        subtype_medians={str(v): float(pp[tt[sub] == v].median()) for v in tt[sub].dropna().unique()
                         if (tt[sub] == v).sum() >= 10},
        er_pos_minus_neg_median=float(er_pos.median() - er_neg.median()) if len(er_pos) and len(er_neg) else None,
        er_mannwhitney_p=float(mannwhitneyu(er_pos, er_neg).pvalue) if len(er_pos) and len(er_neg) else None,
        grade_spearman=float(spearmanr(pp[gm.values], tt.grade[gm])[0]) if gm.sum() > 10 else None)
print(json.dumps(S0, indent=1))
tick("step 0 done")

## Neighbour features and model helpers

In [ ]:
def contact_matrix(D):
    n = len(D["A"])
    C = sp.coo_matrix((np.ones(len(D["ca"]), np.float32), (D["ca"], D["cb"])), shape=(n, n)).tocsr()
    C = (C + C.T).tocsr()
    C.data[:] = 1.0
    return C

def neighbour_means(X, D, C):
    """Mean of X over touching cells (Mc) and over non-touching cells within RING_R of the centroid (Mr)."""
    n, p = X.shape
    Mc = np.full((n, p), np.nan, np.float32)
    Mr = np.full((n, p), np.nan, np.float32)
    nc, nr = np.zeros(n, np.float32), np.zeros(n, np.float32)
    for a, b in zip(D["starts"][:-1], D["starts"][1:]):
        m = b - a
        pr = cKDTree(D["xy"][a:b]).query_pairs(RING_R, output_type="ndarray")
        R = sp.coo_matrix((np.ones(len(pr), np.float32), (pr[:, 0], pr[:, 1])), shape=(m, m)).tocsr()
        R = (R + R.T).tocsr()
        Ci = C[a:b, a:b].tocsr()
        R = (R - R.multiply(Ci)).tocsr()
        R.eliminate_zeros()
        dr, dc = np.asarray(R.sum(1)).ravel(), np.asarray(Ci.sum(1)).ravel()
        Xi = X[a:b]
        Mr[a:b] = np.where(dr[:, None] > 0, (R @ Xi) / np.maximum(dr, 1)[:, None], np.nan)
        Mc[a:b] = np.where(dc[:, None] > 0, (Ci @ Xi) / np.maximum(dc, 1)[:, None], np.nan)
        nr[a:b], nc[a:b] = dr, dc
    return Mc, Mr, nc, nr

def patient_folds(D):
    pats, pcode = np.unique(D["patient"], return_inverse=True)
    folds = [te for _, te in GroupKFold(N_FOLD).split(np.zeros(len(pcode)), groups=pcode)]
    return folds, pcode, pats

def take(blocks, idx):
    return np.hstack([M[idx][:, cols] for M, cols in blocks])

def fit_model(blocks, y, idx):
    m = lgb.LGBMRegressor(random_state=SEED, **LGB)
    m.fit(take(blocks, idx), y[idx])
    return m

def train_subsets(folds, n, rng):
    """One fixed training subsample per fold, shared by every target and model, so ΔR² reflects features only."""
    allidx = np.arange(n)
    return [np.sort(rng.choice(np.setdiff1d(allidx, te, assume_unique=True),
                               min(N_TRAIN, n - len(te)), replace=False)) for te in folds]

def oof(blocks, y, folds, trains):
    pred = np.empty(len(y), np.float32)
    for te, tr in zip(folds, trains):
        pred[te] = fit_model(blocks, y, tr).predict(take(blocks, te))
    return pred

def pat_stats(y, yhat, pcode, n_pat):
    y, yhat = y.astype(np.float64), yhat.astype(np.float64)
    return (np.bincount(pcode, minlength=n_pat).astype(float), np.bincount(pcode, y, n_pat),
            np.bincount(pcode, y ** 2, n_pat), np.bincount(pcode, (y - yhat) ** 2, n_pat))

def boot_r2(W, s):
    n, sy, syy, sse = s
    point = 1 - sse.sum() / (syy.sum() - sy.sum() ** 2 / n.sum())
    N, SY, SYY, SSE = W @ n, W @ sy, W @ syy, W @ sse
    return point, 1 - SSE / (SYY - SY ** 2 / N)

def ci(v):
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

def r_within(y, yhat, img):
    y, yhat = y.astype(np.float64), yhat.astype(np.float64)
    cnt = np.bincount(img)
    yc = y - (np.bincount(img, y) / cnt)[img]
    pc = yhat - (np.bincount(img, yhat) / cnt)[img]
    return float(yc @ pc / np.sqrt((yc @ yc) * (pc @ pc) + 1e-12))

def pearson(a, b):
    ok = np.isfinite(a) & np.isfinite(b)
    return float(np.corrcoef(a[ok], b[ok])[0, 1])

def signal_quality(y):
    q01, q50, q99 = np.percentile(y, [1, 50, 99])
    mad = 1.4826 * np.median(np.abs(y - q50))
    scale = mad if mad > 1e-3 else y.std() + 1e-6
    return dict(pos_frac=float(np.mean((y - q50) / scale > 3)), dyn_range=float(q99 - q50))

def boot_weights(n_pat, rng):
    return rng.multinomial(n_pat, np.full(n_pat, 1 / n_pat), size=N_BOOT).astype(np.float64)

## Step 1: per-marker information map (both cohorts, full panels)

In [ ]:
MAP_ROWS, CTX = [], {}
for key, D in DATA.items():
    rng = np.random.default_rng(SEED)
    folds, pcode, pats = patient_folds(D)
    W = boot_weights(len(pats), rng)
    trains = train_subsets(folds, len(D["A"]), rng)
    C = contact_matrix(D)
    Mc, Mr, nc, nr = neighbour_means(D["A"], D, C)
    CTX[key] = dict(C=C, folds=folds, trains=trains, pcode=pcode, pats=pats, W=W)
    tick(f"{key}: neighbour features")
    extra = [(D["log_area"][:, None], [0])] if D["log_area"] is not None else []
    ci_ = {c: i for i, c in enumerate(D["chan"])}
    targets = [(m, D["A"][:, ci_[m]], m) for m in D["protein"]]
    if key == "jackson":
        targets.append((K27R, D["A"][:, ci_[K27]] - D["A"][:, ci_["histone_h3"]], K27))
    for name, y, drop in targets:
        cols = [i for i in range(len(D["chan"])) if i != ci_[drop]]
        own = [(D["A"], cols)] + extra
        ring = own + [(Mr, cols), (nr[:, None], [0])]
        full = ring + [(Mc, cols), (nc[:, None], [0])]
        preds = {"own": oof(own, y, folds, trains), "own+ring": oof(ring, y, folds, trains),
                 "own+ring+contact": oof(full, y, folds, trains)}
        R2 = {m: boot_r2(W, pat_stats(y, p, pcode, len(pats))) for m, p in preds.items()}
        row = dict(cohort=key, marker=name, group=GROUP_OF[key][name], n_cells=len(y), n_patients=len(pats),
                   shared=name in SHARED or name in SHARED.values(), **signal_quality(y))
        for m, (pt, bv) in R2.items():
            row[f"r2_{m}"], (row[f"r2_{m}_lo"], row[f"r2_{m}_hi"]) = pt, ci(bv)
            row[f"r_within_{m}"] = r_within(y, preds[m], D["img"])
        for lab, a, b in (("ring", "own", "own+ring"), ("contact", "own+ring", "own+ring+contact"),
                          ("full", "own", "own+ring+contact")):
            d = R2[b][1] - R2[a][1]
            row[f"dr2_{lab}"], (row[f"dr2_{lab}_lo"], row[f"dr2_{lab}_hi"]) = R2[b][0] - R2[a][0], ci(d)
        row["context_gain"] = row["dr2_full_lo"] > 0
        row["contact_only"] = row["context_gain"] and not row["dr2_ring_lo"] > 0
        MAP_ROWS.append(row)
        tick(f"{key} {name}: R2 own {row['r2_own']:.3f}, +ring {row['dr2_ring']:+.3f}, +contact {row['dr2_contact']:+.3f}")
    del Mc, Mr
MAP = pd.DataFrame(MAP_ROWS)
MAP.to_csv(f"{OUT}/map_per_marker.csv", index=False)
print(MAP[["cohort", "group", "marker", "r2_own", "r2_own_lo", "dr2_ring", "dr2_ring_lo", "dr2_contact",
           "dr2_full_lo", "context_gain", "contact_only", "r_within_own", "pos_frac"]].round(3).to_string(index=False))

## Gate 1 (Jackson, restricted to Danenberg's channels) and the cross-cohort calibration

In [ ]:
def harmonise(X):
    med = np.median(X, 0)
    q05, q95 = np.percentile(X, [5, 95], 0)
    sc = np.where(q95 - q05 > 1e-3, q95 - q05, X.std(0) + 1e-6)
    return ((X - med) / sc).astype(np.float32)

DJ, DD = DATA["jackson"], DATA["danenberg"]
IN_J = NUC + list(SHARED)
IN_D = NUC + list(SHARED.values())
XJ = harmonise(DJ["A"][:, [DJ["chan"].index(c) for c in IN_J]])
XD = harmonise(DD["A"][:, [DD["chan"].index(c) for c in IN_D]])
_, MrJ, _, nrJ = neighbour_means(XJ, DJ, CTX["jackson"]["C"])
_, MrD, _, nrD = neighbour_means(XD, DD, CTX["danenberg"]["C"])
tick("restricted inputs and ring means")
ALLC = list(range(len(IN_J)))
cj = {c: i for i, c in enumerate(DJ["chan"])}
J_ONLY = [m for m in DJ["protein"] if m not in SHARED] + [K27R]

def jackson_target(m):
    return DJ["A"][:, cj[K27]] - DJ["A"][:, cj["histone_h3"]] if m == K27R else DJ["A"][:, cj[m]]

folds, trains = CTX["jackson"]["folds"], CTX["jackson"]["trains"]
pcode, pats = CTX["jackson"]["pcode"], CTX["jackson"]["pats"]
W = CTX["jackson"]["W"]
rng = np.random.default_rng(SEED + 1)
GATE_ROWS, OOF_J = [], {}
for m in J_ONLY:
    y = jackson_target(m)
    own = [(XJ, ALLC)]
    ring = own + [(MrJ, ALLC), (nrJ[:, None], [0])]
    p_own, p_ring = oof(own, y, folds, trains), oof(ring, y, folds, trains)
    OOF_J[m] = p_own
    a = boot_r2(W, pat_stats(y, p_own, pcode, len(pats)))
    b = boot_r2(W, pat_stats(y, p_ring, pcode, len(pats)))
    row = dict(marker=m, group=GROUP_OF["jackson"][m], r2_own=a[0], r2_own_lo=ci(a[1])[0], r2_own_hi=ci(a[1])[1],
               r2_ring=b[0], dr2_ring=b[0] - a[0], dr2_ring_lo=ci(b[1] - a[1])[0], dr2_ring_hi=ci(b[1] - a[1])[1],
               r_cell=pearson(y, p_own), r_within=r_within(y, p_own, DJ["img"]), **signal_quality(y))
    row["gate1_pass"] = row["r2_own_lo"] > 0
    GATE_ROWS.append(row)
    tick(f"gate 1 {m}: R2 {row['r2_own']:.3f} [{row['r2_own_lo']:.3f}, {row['r2_own_hi']:.3f}]")
GATE = pd.DataFrame(GATE_ROWS)
GATE.to_csv(f"{OUT}/gate1_jackson_restricted.csv", index=False)
print(GATE.round(3).to_string(index=False))

# Cross-cohort calibration on the 11 shared markers (pseudo-targets)
epi_D = DD["epi"]
pcD, patsD = np.unique(DD["patient"], return_inverse=True)[1], np.unique(DD["patient"])
CAL_ROWS = []
for k, s in enumerate(SHARED):
    cols = [i for i in ALLC if i != len(NUC) + k]
    y = XJ[:, len(NUC) + k]
    p_j = oof([(XJ, cols)], y, folds, trains)
    tr = np.sort(rng.choice(len(y), min(N_TRAIN_FINAL, len(y)), replace=False))
    p_d = fit_model([(XJ, cols)], y, tr).predict(take([(XD, cols)], np.arange(len(XD)))).astype(np.float32)
    yd = XD[:, len(NUC) + k]
    pm_true = np.bincount(pcD[epi_D], yd[epi_D], len(patsD)) / np.maximum(np.bincount(pcD[epi_D], minlength=len(patsD)), 1)
    pm_pred = np.bincount(pcD[epi_D], p_d[epi_D], len(patsD)) / np.maximum(np.bincount(pcD[epi_D], minlength=len(patsD)), 1)
    okp = np.bincount(pcD[epi_D], minlength=len(patsD)) >= MIN_TUM
    CAL_ROWS.append(dict(marker=s, danenberg_name=SHARED[s],
                         jackson_cv_r_cell=pearson(y, p_j), jackson_cv_r_within=r_within(y, p_j, DJ["img"]),
                         danenberg_r_cell=pearson(yd, p_d), danenberg_r_within=r_within(yd, p_d, DD["img"]),
                         danenberg_r_patient_mean_tumour=pearson(pm_true[okp], pm_pred[okp])))
    tick(f"calibration {s}: J cv r {CAL_ROWS[-1]['jackson_cv_r_cell']:.3f}, D r {CAL_ROWS[-1]['danenberg_r_cell']:.3f}")
CAL = pd.DataFrame(CAL_ROWS)
CAL.to_csv(f"{OUT}/crosscohort_calibration_shared.csv", index=False)
print(CAL.round(3).to_string(index=False))

## §15-lite: impute Jackson-only markers into Danenberg cells

In [ ]:
PASS = [m for m in GATE[GATE.gate1_pass].marker if m in LITE_TARGETS]
print("lite targets passing gate 1:", PASS)
VIRT_D = {}
tr = np.sort(np.random.default_rng(SEED + 2).choice(len(XJ), min(N_TRAIN_FINAL, len(XJ)), replace=False))
for m in PASS:
    VIRT_D[m] = fit_model([(XJ, ALLC)], jackson_target(m), tr).predict(XD).astype(np.float32)
tick("virtual markers in Danenberg")

def patient_mean(v, D, mask):
    pats, pcode = np.unique(D["patient"], return_inverse=True)
    s = np.bincount(pcode[mask], v[mask], len(pats))
    n = np.bincount(pcode[mask], minlength=len(pats))
    return pd.Series(np.where(n >= MIN_TUM, s / np.maximum(n, 1), np.nan), index=pats)

PF_D = pd.DataFrame({m: patient_mean(v, DD, DD["epi"]) for m, v in VIRT_D.items()})
IN_PM_D = pd.DataFrame({c: patient_mean(XD[:, i], DD, DD["epi"]) for i, c in enumerate(IN_D)})
PF_J_meas = pd.DataFrame({m: patient_mean(jackson_target(m), DJ, DJ["epi"]) for m in PASS})
PF_J_virt = pd.DataFrame({m: patient_mean(OOF_J[m], DJ, DJ["epi"]) for m in PASS})
PF_D.to_csv(f"{OUT}/lite_virtual_patient_means_danenberg.csv")

## §15-lite: patient-level validation against Danenberg labels

In [ ]:
def oof_logit(X, y, seed=SEED):
    P = np.zeros(len(y))
    for r in range(N_REP):
        for tr_, te in StratifiedKFold(5, shuffle=True, random_state=seed + r).split(X, y):
            mdl = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=5000)).fit(X[tr_], y[tr_])
            P[te] += mdl.predict_proba(X[te])[:, 1]
    return P / N_REP

def boot_auc_diff(y, pa, pb, rng):
    out = []
    for _ in range(N_BOOT):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        out.append(roc_auc_score(y[i], pb[i]) - roc_auc_score(y[i], pa[i]))
    return ci(np.array(out))

tD = CLIN["danenberg"][0]
pam = tD.pam50.astype(str).str.lower()
LABELS = {
    "PR+ vs PR- (all)": ((tD.pr == "Positive").astype(float).where(tD.pr.isin(["Positive", "Negative"])), None),
    "PR+ vs PR- (ER+ only)": ((tD.pr == "Positive").astype(float).where(tD.pr.isin(["Positive", "Negative"])),
                              tD.er == "Positive"),
    "basal-like vs rest": ((pam == "basal").astype(float).where(tD.pam50.notna() & (pam != "nan")), None),
    "LumA vs LumB": ((pam == "luma").astype(float).where(pam.isin(["luma", "lumb"])), None)}
VAL_TESTS = [("PR+ vs PR- (all)", "progesterone_receptor_a_b"), ("PR+ vs PR- (ER+ only)", "progesterone_receptor_a_b"),
             ("basal-like vs rest", "keratin_14_krt14"), ("basal-like vs rest", "egfr"),
             ("LumA vs LumB", "gata3"), ("LumA vs LumB", "progesterone_receptor_a_b")]
rng = np.random.default_rng(SEED + 3)
VAL_ROWS = []
for lab, m in VAL_TESTS:
    if m not in PF_D:
        VAL_ROWS.append(dict(test=lab, marker=m, note="failed gate 1"))
        continue
    yv, subset = LABELS[lab]
    df = pd.concat([PF_D[[m]], IN_PM_D], axis=1).join(yv.rename("y"), how="inner")
    if subset is not None:
        df = df[subset.reindex(df.index).fillna(False).values]
    df = df.dropna()
    y = df.y.values.astype(int)
    x = df[m].values
    p_in = oof_logit(df[IN_D].values, y)
    p_both = oof_logit(df[IN_D + [m]].values, y)
    lo, hi = boot_auc_diff(y, p_in, p_both, rng)
    VAL_ROWS.append(dict(test=lab, marker=m, n=len(y), n_pos=int(y.sum()), auc_virtual_alone=roc_auc_score(y, x),
                         auc_inputs=roc_auc_score(y, p_in), auc_inputs_plus_virtual=roc_auc_score(y, p_both),
                         delta_auc=roc_auc_score(y, p_both) - roc_auc_score(y, p_in), delta_lo=lo, delta_hi=hi))
VAL = pd.DataFrame(VAL_ROWS)
VAL.to_csv(f"{OUT}/lite_patient_validation.csv", index=False)
print(VAL.round(3).to_string(index=False))
tick("patient-level validation")

## §15-lite: survival (T1 direction, T2 beyond clinical, T3 beyond the inputs)

In [ ]:
def zs(v):
    return (v - np.nanmean(v)) / (np.nanstd(v) + 1e-12)

def cox_hr(x, C, T, E):
    df = pd.DataFrame(C, columns=[f"c{j}" for j in range(C.shape[1])])
    df["x"] = zs(x)
    df["T"], df["E"] = T, E
    cph = CoxPHFitter(penalizer=0.01).fit(df, "T", "E")
    b, se = cph.params_["x"], cph.standard_errors_["x"]
    return float(b), float(se), float(cph.summary.loc["x", "p"])

def oof_cox(X, T, E):
    risk = np.zeros(len(T))
    for r in range(N_REP):
        for tr_, te in StratifiedKFold(5, shuffle=True, random_state=SEED + r).split(T, E):
            sc = StandardScaler().fit(X[tr_])
            df = pd.DataFrame(sc.transform(X[tr_]), columns=[f"x{j}" for j in range(X.shape[1])])
            df["T"], df["E"] = T[tr_], E[tr_]
            try:
                beta = CoxPHFitter(penalizer=0.1).fit(df, "T", "E").params_.values
            except Exception:
                beta = CoxPHFitter(penalizer=1.0).fit(df, "T", "E").params_.values
            h = sc.transform(X[te]) @ beta
            risk[te] += (h - h.mean()) / (h.std() + 1e-12)
    return risk / N_REP

def cidx(T, E, risk):
    return concordance_index(T, -risk, E)

def boot_dc(T, E, ra, rb, rng):
    out = []
    for _ in range(N_BOOT):
        i = rng.integers(0, len(T), len(T))
        if E[i].sum() < 2:
            continue
        out.append(cidx(T[i], E[i], rb[i]) - cidx(T[i], E[i], ra[i]))
    out = np.array(out)
    p = max(2 * min((out <= 0).mean(), (out >= 0).mean()), 1 / len(out))
    return ci(out) + (float(p),)

def bh(p):
    p = np.asarray(p, float)
    o = np.argsort(p)
    q = p[o] * len(p) / (np.arange(len(p)) + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[o] = np.minimum(q, 1)
    return out

tD, covD = CLIN["danenberg"]
dfD = pd.concat([PF_D, IN_PM_D.add_prefix("in_"), covD], axis=1).join(tD[["dss_time", "dss_event"]], how="inner")
dfD = dfD.dropna(subset=list(covD.columns) + ["dss_time", "dss_event"] + [f"in_{c}" for c in IN_D])
TD = np.maximum(dfD.dss_time.values.astype(float), 0.1)
ED = dfD.dss_event.values.astype(float)
CD = dfD[list(covD.columns)].values.astype(float)
IND = dfD[[f"in_{c}" for c in IN_D]].values.astype(float)
print(f"Danenberg DSS: {len(TD)} patients, {int(ED.sum())} events")
tJ, covJ = CLIN["jackson"]
rng = np.random.default_rng(SEED + 4)
r_clin = oof_cox(CD, TD, ED)
r_clin_in = oof_cox(np.c_[CD, IND], TD, ED)
SURV_ROWS = []
for m in PASS:
    ok = dfD[m].notna().values
    x = dfD[m].values
    row = dict(marker=m, group=GROUP_OF["jackson"][m], n_danenberg=int(ok.sum()), events_danenberg=int(ED[ok].sum()),
               prior_sign=PRIOR_SIGN.get(m))
    b, se, p = cox_hr(x[ok], CD[ok], TD[ok], ED[ok])
    row.update(hr_danenberg_virtual=np.exp(b), hr_dv_lo=np.exp(b - 1.96 * se), hr_dv_hi=np.exp(b + 1.96 * se), p_dv=p)
    for lab, PF in (("jackson_measured", PF_J_meas), ("jackson_virtual", PF_J_virt)):
        dj = pd.concat([PF[[m]], covJ], axis=1).join(tJ[["dss_time", "dss_event"]], how="inner").dropna()
        bj, sej, pj = cox_hr(dj[m].values, dj[list(covJ.columns)].values.astype(float),
                             np.maximum(dj.dss_time.values.astype(float), 0.1), dj.dss_event.values.astype(float))
        row.update({f"hr_{lab}": np.exp(bj), f"hr_{lab}_lo": np.exp(bj - 1.96 * sej),
                    f"hr_{lab}_hi": np.exp(bj + 1.96 * sej), f"p_{lab}": pj, f"n_{lab}": len(dj),
                    f"events_{lab}": int(dj.dss_event.sum())})
    row["t1_sign_matches_jackson_measured"] = bool(np.sign(np.log(row["hr_danenberg_virtual"])) ==
                                                   np.sign(np.log(row["hr_jackson_measured"])))
    row["t1_sign_matches_prior"] = (None if row["prior_sign"] is None else
                                    bool(np.sign(np.log(row["hr_danenberg_virtual"])) == row["prior_sign"]))
    xf = np.where(ok, x, np.nanmean(x))
    r_cx = oof_cox(np.c_[CD, xf], TD, ED)
    r_cix = oof_cox(np.c_[CD, IND, xf], TD, ED)
    lo, hi, p2 = boot_dc(TD, ED, r_clin, r_cx, rng)
    row.update(c_clin=cidx(TD, ED, r_clin), c_clin_x=cidx(TD, ED, r_cx), t2_dc=cidx(TD, ED, r_cx) - cidx(TD, ED, r_clin),
               t2_lo=lo, t2_hi=hi, t2_p=p2)
    lo, hi, p3 = boot_dc(TD, ED, r_clin_in, r_cix, rng)
    row.update(c_clin_in=cidx(TD, ED, r_clin_in), c_clin_in_x=cidx(TD, ED, r_cix),
               t3_dc=cidx(TD, ED, r_cix) - cidx(TD, ED, r_clin_in), t3_lo=lo, t3_hi=hi, t3_p=p3)
    SURV_ROWS.append(row)
    tick(f"survival {m}: HR D-virtual {row['hr_danenberg_virtual']:.2f}, J-measured {row['hr_jackson_measured']:.2f}, "
         f"T2 {row['t2_dc']:+.3f}, T3 {row['t3_dc']:+.3f}")
SURV = pd.DataFrame(SURV_ROWS)
if len(SURV):
    SURV["t2_q"], SURV["t3_q"] = bh(SURV.t2_p), bh(SURV.t3_p)
SURV.to_csv(f"{OUT}/lite_survival.csv", index=False)
print(SURV.round(3).to_string(index=False))

## Figures

In [ ]:
def label_groups(ax, key, present):
    pos = 0
    for j, (grp, ms) in enumerate(GROUPS[key].items()):
        k = sum(m in present for m in ms)
        if k:
            ax.axvline(pos - 0.5, color="grey", lw=0.5)
            ax.text(pos - 0.4, 1.02 + 0.06 * (j % 2), grp, transform=ax.get_xaxis_transform(), fontsize=7, va="bottom")
            pos += k

COL = {"danenberg": ("#a1d99b", "#41ab5d", "#006d2c"), "jackson": ("#9ecae1", "#4292c6", "#08519c")}
NAME = {"danenberg": "Danenberg 2022", "jackson": "Jackson 2020"}
fig, axes = plt.subplots(2, 1, figsize=(18, 9))
for ax, key in zip(axes, ("danenberg", "jackson")):
    g = MAP[MAP.cohort == key].copy()
    present = set(g.marker)
    order = [m for grp in GROUPS[key] for m in GROUPS[key][grp] if m in present]
    g = g.set_index("marker").loc[order]
    x = np.arange(len(g))
    for j, (mdl, w) in enumerate((("own", -0.27), ("own+ring", 0.0), ("own+ring+contact", 0.27))):
        v = g[f"r2_{mdl}"].values
        err = np.vstack([v - g[f"r2_{mdl}_lo"].values, g[f"r2_{mdl}_hi"].values - v])
        ax.bar(x + w, v, 0.27, color=COL[key][j], yerr=err, capsize=1.5, error_kw=dict(lw=0.6),
               label=f"{NAME[key]}: {mdl}")
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=70, ha="right", fontsize=7)
    ax.set_xlim(-0.6, len(g) - 0.4)
    ax.set_ylabel(f"{NAME[key]}\ncell-level R² (5-fold, patient-grouped)\nwhiskers: 95% patient-bootstrap CI", fontsize=8)
    label_groups(ax, key, present)
    ax.legend(loc="upper left", bbox_to_anchor=(1.005, 1), fontsize=7, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(f"{OUT}/fig_map.png", dpi=160, bbox_inches="tight")
plt.close(fig)

fig, axes = plt.subplots(2, 1, figsize=(18, 8))
for ax, key in zip(axes, ("danenberg", "jackson")):
    g = MAP[MAP.cohort == key].copy()
    present = set(g.marker)
    order = [m for grp in GROUPS[key] for m in GROUPS[key][grp] if m in present]
    g = g.set_index("marker").loc[order]
    x = np.arange(len(g))
    for j, (lab, w, name) in enumerate((("ring", -0.2, "non-touching cells within 30 µm"),
                                        ("contact", 0.2, "touching cells, added on top"))):
        v = g[f"dr2_{lab}"].values
        err = np.vstack([v - g[f"dr2_{lab}_lo"].values, g[f"dr2_{lab}_hi"].values - v])
        ax.bar(x + w, v, 0.4, color=COL[key][j + 1], yerr=err, capsize=1.5, error_kw=dict(lw=0.6),
               label=f"{NAME[key]}: ΔR², {name}")
    ax.axhline(0, color="black", lw=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=70, ha="right", fontsize=7)
    ax.set_xlim(-0.6, len(g) - 0.4)
    ax.set_ylabel(f"{NAME[key]}\nΔ cell-level R² from neighbours\nwhiskers: 95% patient-bootstrap CI", fontsize=8)
    label_groups(ax, key, present)
    ax.legend(loc="upper left", bbox_to_anchor=(1.005, 1), fontsize=7, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(f"{OUT}/fig_context.png", dpi=160, bbox_inches="tight")
plt.close(fig)

if len(SURV):
    fig, ax = plt.subplots(figsize=(7, 0.45 * len(SURV) + 1.2))
    y0 = np.arange(len(SURV))
    for off, lab, col, name in ((-0.15, "danenberg_virtual", "#41ab5d", "Danenberg 2022, virtual (DSS, adjusted)"),
                                (0.15, "jackson_measured", "#4292c6", "Jackson 2020 Basel, measured (DSS, adjusted)")):
        hr = SURV[f"hr_{lab}"].values
        lo = SURV["hr_dv_lo" if lab == "danenberg_virtual" else f"hr_{lab}_lo"].values
        hi = SURV["hr_dv_hi" if lab == "danenberg_virtual" else f"hr_{lab}_hi"].values
        ax.errorbar(hr, y0 + off, xerr=[hr - lo, hi - hr], fmt="o", color=col, ms=4, lw=1, label=name)
    ax.axvline(1, color="grey", lw=0.6)
    ax.set_xscale("log")
    ax.set_yticks(y0)
    ax.set_yticklabels(SURV.marker)
    ax.set_xlabel("hazard ratio per SD of the tumour-cell mean (95% CI)")
    ax.legend(loc="lower left", bbox_to_anchor=(0, 1.01), fontsize=7, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(f"{OUT}/fig_lite_survival.png", dpi=160, bbox_inches="tight")
    plt.close(fig)
tick("figures")

## Pre-registered answers

In [ ]:
ans = {"step0": S0}
for key in DATA:
    g = MAP[MAP.cohort == key]
    ans[f"map_{key}"] = dict(
        n_markers=len(g),
        context_gain=g[g.context_gain].marker.tolist(),
        context_gain_contact_only=g[g.contact_only].marker.tolist(),
        context_gain_ring=g[g.dr2_ring_lo > 0].marker.tolist(),
        r2_own_by_group=g.groupby("group").r2_own.median().round(3).to_dict(),
        dr2_full_by_group=g.groupby("group").dr2_full.median().round(4).to_dict())
ans["GK_section14_reading"] = {k: len(ans[f"map_{k}"]["context_gain"]) >= 2 for k in DATA}
ans["gate1_pass"] = GATE[GATE.gate1_pass].marker.tolist()
ans["gate1_fail"] = GATE[~GATE.gate1_pass].marker.tolist()
ans["lite_targets"] = PASS
ans["calibration_median"] = CAL[["jackson_cv_r_cell", "danenberg_r_cell", "danenberg_r_patient_mean_tumour"]].median().round(3).to_dict()
if len(VAL):
    ans["validation"] = VAL.round(3).to_dict(orient="records")
if len(SURV):
    ans["survival"] = dict(
        t1_sign_matches_jackson=SURV.set_index("marker").t1_sign_matches_jackson_measured.to_dict(),
        t1_sign_matches_prior=SURV.set_index("marker").t1_sign_matches_prior.to_dict(),
        t2_significant_bh=SURV[SURV.t2_q < 0.05].marker.tolist(),
        t3_significant_bh=SURV[SURV.t3_q < 0.05].marker.tolist())
json.dump(ans, open(f"{OUT}/summary.json", "w"), indent=1, default=str)
print(json.dumps(ans, indent=1, default=str))
tick("done")